<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# Regression and Classification: Evaluating Classification Models
---
<small>© COPYRIGHT 2026 DECODED LIMITED. ALL RIGHTS RESERVED. Web: https://decoded.com/ </small>


---

### Summary

In this chapter we explore how classification models are evaluated and compared in real-world machine learning workflows. Through a combination of theory, hands-on exercises, and practical examples, we will examine how to train multiple classification models, validate their performance using cross-validation, and optimise them using hyperparameter tuning. We will focus particularly on evaluating models in imbalanced datasets using precision and recall, comparing models against baseline performance, and interpreting results in a policy or decision-making context.


### Learning Objectives

* Understand how classification models are trained and evaluated in a machine learning workflow
* Recognise the challenges posed by **imbalanced datasets** and why accuracy can be misleading
* Train multiple classification models including logistic regression, decision trees, and random forests
* Apply **train–test splitting** and understand the importance of evaluating models on unseen data
* Use **cross-validation** to estimate model performance and assess model stability
* Compare models using appropriate metrics such as **precision and recall**
* Perform **hyperparameter tuning using GridSearchCV** to optimise model performance
* Understand the purpose of **baseline models** and compare trained models against random prediction
* Interpret classification model performance using confusion matrices and evaluation metrics
* Connect model evaluation results to **practical decision-making contexts**



### Overview

[**Understanding the classification problem**](#classification_problem)

* What is the prediction task?
* Why is this a classification problem?
* What challenges arise when the dataset is imbalanced?
* *Challenge*: Explore the class distribution

[**Preparing data for model training**](#data_preparation)

* Why do we split data into training and test sets?
* What is stratified sampling?
* How does train–test splitting prevent data leakage?
* *Demo*: Use `train_test_split()` in `sklearn`

[**Training classification models**](#model_training)

* What are logistic regression, decision trees, and random forests?
* Why train multiple models?
* How do models learn patterns from training data?
* *Demo*: Train models using `sklearn` classifiers

[**Evaluating models with cross-validation**](#cross_validation)

* What is cross-validation?
* Why do we evaluate models across multiple folds?
* How do we interpret variation in model performance?
* *Challenge*: Compare cross-validation scores

[**Improving models with hyperparameter tuning**](#hyperparameter_tuning)

* What are hyperparameters?
* How does grid search help improve model performance?
* How does `GridSearchCV` work?
* *Demo*: Tune classification models using `GridSearchCV`

[**Comparing models objectively**](#model_comparison)

* How do we compare multiple trained models?
* Why visualise cross-validation scores?
* What does a boxplot of model performance tell us?
* *Challenge*: Compare model stability and performance

[**Using baseline models for context**](#baseline_models)

* What is a baseline model?
* Why compare models against random predictions?
* How do we implement a `DummyClassifier`?
* *Challenge*: Evaluate baseline performance

[**Evaluating the final model**](#final_model_evaluation)

* Why must the final model be evaluated on the test set?
* What are precision and recall?
* How do we interpret confusion matrix outcomes?
* *Challenge*: Interpret model predictions in context



### Introductions

#### Purpose: What is the aim of the content?

The purpose of this workshop is to develop a clear understanding of how classification models are evaluated and compared in practice. While training models is an important step in machine learning, selecting the most appropriate model requires careful evaluation using appropriate metrics and validation techniques. In this lesson, learners will explore how to compare multiple models, apply cross-validation, optimise models through hyperparameter tuning, and interpret performance metrics such as precision and recall. These skills are essential for building reliable models that support real-world decision-making.



#### Objectives: What will I learn?

* Understand the workflow for training and evaluating classification models
* Recognise the impact of class imbalance on model evaluation
* Train and compare multiple classification algorithms
* Apply cross-validation to estimate model performance
* Perform hyperparameter tuning using grid search
* Evaluate models using precision and recall metrics
* Implement baseline models for meaningful comparison
* Interpret classification outcomes using confusion matrices
* Connect model evaluation results to real-world decision contexts


#### Process: How will I learn this content?

Throughout the workshop, we will:

* explore concepts through short explanations and examples
* train and evaluate models using Python and `scikit-learn`
* compare model performance using visualisations and metrics
* complete guided exercises and challenges
* discuss how evaluation results inform real-world decisions



#### Applications: Why study this?

Classification models are widely used to support decision-making across many domains, including finance, healthcare, public policy, marketing, and social research. However, selecting a model based solely on accuracy can lead to misleading conclusions, particularly when datasets are imbalanced. Understanding how to properly evaluate classification models using metrics such as precision and recall allows analysts to assess the real impact of model predictions. These evaluation techniques help ensure that machine learning models provide meaningful insights and reliable support for policy, business, and operational decisions.





---
---
# Problem Statement
---
---




We are building a **classification model** to predict whether an individual or household is likely to experience **residential out-migration**. In this context, the target variable (Y) represents a binary outcome:

* **(Y = 1)** indicates that a person or household leaves the area (out-migration)
* **(Y = 0)** indicates that they remain in the area

The model uses a range of explanatory features to estimate the probability that an individual belongs to one of these two categories. These features may include demographic, economic, or housing-related characteristics that help explain patterns of residential mobility.

The purpose of this model is not simply to make predictions, but to **support policy decision-making**. In particular, the model is designed to help identify groups or areas where the risk of out-migration is higher. These insights can then be used to inform **targeted interventions under the Programme for Government**, such as initiatives aimed at retaining graduates, supporting housing stability, or strengthening regional economic opportunities.

Because policy resources are limited, the model must be evaluated carefully to ensure that interventions are directed toward individuals who are genuinely at risk of leaving. This is why metrics such as **precision and recall** become important. Precision helps ensure that resources are not wasted on individuals who would not have left anyway, while recall measures how effectively the model identifies those who are truly at risk.

In this lesson, we will explore how classification models can be trained, validated, and evaluated in order to support this type of evidence-based policy decision-making.

---
---
# Selecting Our Evaluation Metric For This Problem
---
---


## Policy Context

We are predicting **residential out-migration (Y=1)** to inform targeted interventions under the Programme for Government.

Public funds are limited. Every intervention must be defensible.

---
## Cost of Errors
---

### False Positive (FP)

We intervene, but the resident would have stayed anyway.

**Cost:**

* Deadweight loss (wasted subsidy)
* Reduced funds for genuine cases
* Audit and accountability risk

This is an **immediate, measurable fiscal cost**.

### False Negative (FN)

We fail to intervene, and the resident leaves.

**Cost:**

* Long-term economic and demographic impact
* Harder to measure directly in the short term

This is a **strategic, longer-term cost**.

---
---

## Why Precision Is Prioritised
---
---

Precision answers:

> “When we intervene, how often are we correct?”

High precision ensures:

* Interventions are tightly targeted
* Public money is not wasted
* Decisions withstand scrutiny

In a budget-constrained public sector environment, avoiding expenditure on ineffective interventions is crucial.

---
## Policy Trade-Off
---

We accept:

* Some at-risk residents may not be identified (lower recall)

In exchange for:

* High confidence that funded interventions are justified

This is a deliberate choice to prioritise **resource efficiency and accountability**.



## Imports for the Notebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_recall_curve, auc
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

### Importing the Dataset

##  Looking at the Class Distribution


It appears that the dataset is imbalanced, so as mentioned earlier the - accuracy is not a good metric to evaluate the model performance. We will use precision as our primary metric to evaluate the model performance.

## Train/Test Split


Now we will split the data into a training set and a test set. The training set will be used to train the model, and the test set will be used to evaluate the model's performance on unseen data. Remember:

- stratify the split to maintain the class distribution in both sets. Since the dataset is imbalanced
- do not toouch the test set in any way until the final evaluation step. The test set should only be used once to evaluate the final model performance.

## Model Fitting


Let us first instantiate our classifcation models

we will fit our models

In the next section we jump straight to cross validation as the single train/test split is not enough to evaluate the models. (remeber, our  model might get lucky with the train/test split so we do k-folds cross validation to get a better estimate of the model performance)

## Cross-Validation


### unlike regression we have to consider class balance so we will use stratified k folds instead of the usual kfolds

### Rather than try to manually cross reference the precision scores across the different models, we can use boxplots to visualise the distribution of precision scores across the different models.

Based on intital fits, the SVC classifier appears to have a much harder time being consistent in its precision scores. But remember, the intial fits are just that - initial. We will need to do hyperparameter tuning to see if we can improve the performance of our models

# Hyperparameter Tuning

In [107]:
param_grid_lr = {
    'C': [0.01, 0.1, 10],   
    'solver': ['liblinear']        
}

param_grid_dt = {
    'max_depth': [None, 5],       
}

param_grid_rf= {
    'n_estimators': [3,5,8],}



---
##  Model Evaluation For Classification
---

how do we know that are model has learned anything at all? a great way is to create a baseline model model that just makes random prediction and see how our model performs against that baseline.

# Policy Interpretation

Random guessing results show that:

when the model predicts a resident will leave, it is correct only about 17% of the time (precision ~17%).

Only ~17% of true leavers would be identified.

Most funding would be misdirected.

This is not a viable policy tool.

---
# Evaluating The Best Model
---

we import the best model from the previous section


# Precision

When the model predicts that someone is at risk of leaving, it is correct about 97% of the time.

So the misallocation of financial resources is very low.

### Policy framing

The model results in very little wasted intervention spending.

# Recall

The model correctly identifies about 61% of the individuals who actually leave.

### policy framing

The model captures most at-risk individuals but does not detect all of them.

The remaining 37% missed may still leave without intervention.

### So the trade-off is clear

The model prioritises accuracy of intervention over identifying every possible leaver. As we'd hoped, the model is very good at targeting interventions to those who are truly at risk, but it does not capture every single case. This is a common and acceptable trade-off in many real-world applications, especially when resources are limited and the cost of false positives is high.


---
# Further Consideration: Adding Uncertainty Estimates To Our Predictions
---

As we mentioned in the Regression notebook, using a single value (point estimate) to evaluate model performance can be misleading. we use bootstrapping to measure the uncertainty in precision scores to get a better understanding of the model performance in the case the test set is slightly different than the one we have.

In simple terms, this means:

If we tested the model on another similar dataset, the precision might change a little.
The confidence interval shows how much it could realistically change.

So instead of saying:
The model is correct exactly 97% of the time,

Our test set is just one sample of data.

If you collected another similar dataset:
- Precision might be 95%
- or 96%
- or 98%

The interval shows the likely range.

we say:
It is probably between 95% and 99% correct, depending on the sample we happened to test it on.



---
# Further Considerations - Thresholding
---

At a threshold of 0.5 default threshold. the model achieves precision of 0.96, meaning only 4% of funded interventions would be misallocated. This reflects strong fiscal discipline. Lets us look at at an example of how we can improve of test accuracy by tuning the threshold